# Figure 3 · Choose the primary evaluation protocol and test ESM-nonlinear OOD stability

> Read-only: `work/results/probe_protocol_full_selected_k.csv` and Fig2 `cross_split_similarity.csv`. Cluster OOD k is fixed at 12. This figure does not train models; four-protocol × three-method AUROCs are frozen from `probe_protocol_full_selected_k.py` / the protocol table.

Core argument chain:

- **3A｜Standardized evaluation pressure**: compare how strict the three non-random protocols are for the binary baseline.
- **3B｜AUROC drop from Random**: performance drop from Random to three strict protocols; check whether ESM-nonlinear drops less.
- **3C｜Subtype OOD method comparison**: under the chosen primary protocol, compare three methods' per-drug AUROC and mean CI on Subtype OOD.
- **3D｜Random CV overestimates a subset of drugs**: under ESM-nonlinear, show per-drug `Random CV − Subtype OOD` AUROC deltas—most drugs change little, a minority are clearly overestimated by Random CV.

Conclusion goal: Subtype OOD is strongest on subtype-shift and AUROC stress, so it is the primary evaluation scheme; under that scheme, ESM-nonlinear keeps the highest OOD performance, while Random CV mainly overstates reliability for a minority of drug tasks.


In [ ]:
from __future__ import annotations
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / "work" / "results").exists() else CWD.parent
RES = PROJECT_ROOT / "work" / "results"
FIG2_OUT = PROJECT_ROOT / "results" / "notebooks" / "fig2"
FIG_DIR = PROJECT_ROOT / "figures" / "manuscript" / "fig3"
OUT_DIR = PROJECT_ROOT / "results" / "notebooks" / "fig3"
# Cluster OOD k fixed contract; CSV no longer required for main pipeline.
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

SURFACE, INK, INK2, INK3, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#8a8984", "#e6e5e1"
M_COLOR = {"binary": "#8a8984", "esm_lr": "#2a78d6", "esm_lgb": "#eb6834"}
M_LABEL = {"binary": "binary mutation", "esm_lr": "ESM-linear", "esm_lgb": "ESM-nonlinear"}
PROTO_LABEL = {"random": "Random", "rg": "Patient", "clu": "Cluster", "ood": "Subtype"}
STRICT_ORDER = ["rg", "clu", "ood"]
METHODS = ["binary", "esm_lr", "esm_lgb"]

mpl.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "font.size": 11,
    "axes.edgecolor": GRID,
    "axes.labelcolor": INK2,
    "text.color": INK,
    "xtick.color": INK2,
    "ytick.color": INK2,
    "axes.grid": False,
    "font.family": "DejaVu Sans",
})

K_STAR = 12
protocol_path = RES / "probe_protocol_full_selected_k.csv"
if not protocol_path.exists():
    raise FileNotFoundError(
        f"Missing {protocol_path}. Run `uv run --python venv/bin/python python probe_protocol_full_selected_k.py` first."
    )
auc = pd.read_csv(protocol_path)
leak = pd.read_csv(FIG2_OUT / "cross_split_similarity.csv")

required = [f"{p}_{m}" for p in ["random", "rg", "clu", "ood"] for m in METHODS]
missing = [c for c in required if c not in auc.columns]
if missing:
    raise ValueError(f"{protocol_path.name} is missing required columns: {missing}")

def boot_ci(vals, n=2000, seed=42):
    vals = np.asarray(vals, float)
    vals = vals[~np.isnan(vals)]
    if len(vals) < 2:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    bs = [rng.choice(vals, len(vals), replace=True).mean() for _ in range(n)]
    return float(vals.mean()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))

summary_rows = []
for proto in ["random", *STRICT_ORDER]:
    for method in METHODS:
        mean, lo, hi = boot_ci(auc[f"{proto}_{method}"].values)
        summary_rows.append({
            "protocol": proto,
            "protocol_label": PROTO_LABEL[proto],
            "method": method,
            "method_label": M_LABEL[method],
            "mean": mean,
            "ci_low": lo,
            "ci_high": hi,
        })
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT_DIR / "protocol_method_auc_summary.csv", index=False)
print("fig3 setup |", len(auc), "drugs | selected k =", K_STAR, "| protocol source:", protocol_path.name)
print(summary.pivot(index="protocol_label", columns="method_label", values="mean").loc[["Random", "Patient", "Cluster", "Subtype"]].round(3))

In [ ]:
# 3A: standardized evaluation-pressure map. Per column, percentile-rank across three non-random protocols; higher = stricter / more OOD.
# Load precomputed data (from notebooks/scripts/fig3_protocol_summary.py)
stress        = pd.read_csv(OUT_DIR / "protocol_pressure_metrics.csv")
stress_scaled = pd.read_csv(OUT_DIR / "protocol_pressure_scaled.csv")

metric_cols   = ["anti_leakage", "sequence_novelty", "subtype_shift", "auroc_stress"]
metric_labels = ["Anti-leakage", "Sequence novelty", "Subtype shift", "AUROC stress"]
protocol_order = ["Patient", "Cluster", "Subtype"]

fig, ax = plt.subplots(figsize=(7.8, 3.7))
protocol_order = ["Patient", "Cluster", "Subtype"]
mat = stress_scaled.set_index("protocol_label").loc[protocol_order, metric_cols].values
im = ax.imshow(mat, cmap="YlGnBu", vmin=0, vmax=1, aspect="auto")

ax.set_yticks(np.arange(len(protocol_order)))
ax.set_yticklabels(protocol_order, fontsize=10)
ax.set_xticks(np.arange(len(metric_cols)))
ax.set_xticklabels(metric_labels, rotation=22, ha="right", fontsize=9)

for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, f"{mat[i, j]:.2f}", ha="center", va="center", fontsize=9,
                color=INK if mat[i, j] < 0.62 else SURFACE, fontweight="bold" if mat[i, j] >= 0.74 else "normal")

ax.set_title("Fig 3A - Standardized evaluation pressure across non-random protocols",
             loc="left", color=INK, fontsize=11.5, pad=10)
for s in ("top", "right", "bottom", "left"):
    ax.spines[s].set_visible(False)
ax.tick_params(length=0)
cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label("percentile rank (0–1)", fontsize=9)
fig.text(0.01, 0.01,
         "Each column is converted to percentile rank across Patient / Cluster / Subtype; larger values indicate stronger evaluation pressure.",
         color=INK3, fontsize=8.5)
fig.tight_layout(rect=[0, 0.05, 1, 1])
fig.savefig(FIG_DIR / "fig3A_protocol_pressure.png", dpi=150, bbox_inches="tight")
print("saved fig3A_protocol_pressure.png")
plt.show()

In [ ]:
# 3B: AUROC drop from Random to strict protocols; directly quantifies who is more stable.
# Load precomputed data (from notebooks/scripts/fig3_protocol_summary.py)
drop_summary = pd.read_csv(OUT_DIR / "random_to_strict_protocol_drop.csv")

fig, ax = plt.subplots(figsize=(7.8, 4.4))
x = np.arange(len(STRICT_ORDER)); w = 0.23
for j, method in enumerate(METHODS):
    rows = drop_summary[drop_summary["method"] == method].set_index("protocol").loc[STRICT_ORDER]
    y = rows["mean_drop"].values
    yerr = np.vstack([y - rows["ci_low"].values, rows["ci_high"].values - y])
    xpos = x + (j - 1) * w
    ax.bar(xpos, y, w, color=M_COLOR[method], edgecolor=SURFACE, label=M_LABEL[method])
    ax.errorbar(xpos, y, yerr=yerr, fmt="none", ecolor=INK2, elinewidth=1.0, capsize=2.5)
    for xi, yi in zip(xpos, y):
        ax.text(xi, yi + 0.004, f"{yi:.3f}", ha="center", va="bottom", fontsize=8, color=INK2)
ax.axhline(0, color=INK2, lw=1)
ax.set_xticks(x)
ax.set_xticklabels([PROTO_LABEL[p] for p in STRICT_ORDER], fontsize=10)
ax.set_ylabel("AUROC drop from Random")
ax.set_title("Fig 3B - Performance drop under stricter evaluation protocols",
             loc="left", color=INK, fontsize=11.5, pad=10)
ax.legend(frameon=False, fontsize=8.5, loc="upper left")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3B_random_to_strict_drop.png", dpi=150, bbox_inches="tight")
print("saved fig3B_random_to_strict_drop.png")
plt.show()

In [ ]:
# 3C: after choosing Subtype OOD as primary, compare three methods' per-drug AUROC and mean CI.
# Load precomputed data (from notebooks/scripts/fig3_protocol_summary.py)
subtype = pd.read_csv(OUT_DIR / "subtype_ood_method_auc.csv")

fig, ax = plt.subplots(figsize=(6.6, 4.8))
x = np.arange(len(METHODS))
rng = np.random.default_rng(42)
for j, method in enumerate(METHODS):
    vals = subtype.loc[subtype["method"] == method, "auroc"].values
    jitter = rng.normal(0, 0.035, len(vals))
    ax.scatter(np.full(len(vals), j) + jitter, vals, s=30, color=M_COLOR[method],
               alpha=0.34, edgecolor="none", zorder=2)
    mean, lo, hi = boot_ci(vals)
    ax.errorbar(j, mean, yerr=[[mean - lo], [hi - mean]], fmt="o", ms=10,
                color=M_COLOR[method], markeredgecolor=SURFACE, capsize=4, zorder=4)
    ax.text(j, hi + 0.008, f"{mean:.3f}", ha="center", va="bottom", fontsize=9, color=INK)
for _, g in subtype.pivot(index=["class", "drug"], columns="method", values="auroc").reset_index().iterrows():
    ax.plot(x, [g[m] for m in METHODS], color=GRID, lw=0.7, alpha=0.45, zorder=1)
ax.set_xticks(x)
ax.set_xticklabels([M_LABEL[m] for m in METHODS], fontsize=9)
ax.set_ylabel("Subtype OOD AUROC")
ax.set_title("Fig 3C - Model performance under the primary Subtype OOD test",
             loc="left", color=INK, fontsize=11.5, pad=10)
ax.set_ylim(max(0.5, subtype["auroc"].min() - 0.04), min(1.02, subtype["auroc"].max() + 0.05))
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3C_subtype_ood_methods.png", dpi=150, bbox_inches="tight")
print("saved fig3C_subtype_ood_methods.png")
plt.show()

In [ ]:
# 3D: ranked per-drug ESM-nonlinear AUROC delta (Random CV − Subtype OOD).
# Load precomputed data (from notebooks/scripts/fig3_protocol_summary.py)
drop_df = pd.read_csv(OUT_DIR / "random_minus_subtype_esm_nonlinear_drop.csv")

fig, ax = plt.subplots(figsize=(8.4, 0.34 * len(drop_df) + 1.7))
y = np.arange(len(drop_df))
colors = np.where(drop_df["drop"] > 0.05, "#eb6834", "#b8b6b0")
ax.barh(y, drop_df["drop"], color=colors, edgecolor=SURFACE, height=0.68)
ax.axvline(0, color=INK2, lw=1.0)
ax.axvline(0.05, color="#eb6834", lw=1.1, ls="--")
ax.text(0.052, len(drop_df) - 0.4, "drop > 0.05", color="#eb6834", fontsize=8.5,
        ha="left", va="top")

for yi, r in drop_df.iterrows():
    if r["drop"] > 0.05:
        ax.text(r["drop"] + 0.006, yi, f"{r['drug']} {r['drop']:+.3f}",
                va="center", ha="left", fontsize=8, color=INK)

ax.set_yticks(y)
ax.set_yticklabels([f"{r['class']}/{r['drug']}" for _, r in drop_df.iterrows()], fontsize=8)
ax.set_xlabel("AUROC drop: Random CV − Subtype OOD (ESM-nonlinear)")
ax.set_title("Fig 3D - Random CV overestimates a subset of drug tasks",
             loc="left", color=INK, fontsize=11.5, pad=10)
mean_drop = drop_df["drop"].mean()
large_drop = int((drop_df["drop"] > 0.05).sum())
stable = len(drop_df) - large_drop
ax.text(0.02, 0.96,
        f"Mean drop = {mean_drop:.3f}\nStable or small change: {stable}/{len(drop_df)}\nOverestimated: {large_drop}/{len(drop_df)}",
        transform=ax.transAxes, ha="left", va="top", fontsize=9, color=INK2,
        bbox=dict(boxstyle="round,pad=0.35", facecolor=SURFACE, edgecolor=GRID, alpha=0.92))
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3D_random_minus_subtype_drop.png", dpi=150, bbox_inches="tight")
print("saved fig3D_random_minus_subtype_drop.png")
print(drop_df[["class", "drug", "random_esm_lgb", "ood_esm_lgb", "drop", "status"]].sort_values("drop", ascending=False).head(10).round(3))
plt.show()

## Output artifacts

| Panel | Figure | Table |
|---|---|---|
| 3A | `figures/manuscript/fig3/fig3A_protocol_pressure.png` | `results/notebooks/fig3/protocol_pressure_metrics.csv`, `protocol_pressure_scaled.csv` |
| 3B | `figures/manuscript/fig3/fig3B_random_to_strict_drop.png` | `results/notebooks/fig3/random_to_strict_protocol_drop.csv` |
| 3C | `figures/manuscript/fig3/fig3C_subtype_ood_methods.png` | `results/notebooks/fig3/subtype_ood_method_auc.csv` |
| 3D | `figures/manuscript/fig3/fig3D_random_minus_subtype_drop.png` | `results/notebooks/fig3/random_minus_subtype_esm_nonlinear_drop.csv` |

Suggested caption takeaway: Subtype OOD has the strongest deployment-relevant evaluation pressure among the non-random protocols. Under this primary test, ESM-nonlinear achieves the highest mean AUROC overall. Comparing Random CV against Subtype OOD shows that most drug tasks change little, but a subset is substantially overestimated by Random CV.

## Combined Figure 3 · panels A–D

Redraw A–D directly from intermediate tables and in-memory variables instead of reading single-panel PNGs, so fonts, sizes, line widths, and colors stay consistent in the combined figure.


In [ ]:
# Combined Figure 3: redraw A-D directly from data/tables, not from saved PNGs.
TEXT_COLOR = "#000000"
PANEL_TITLE_SIZE = 13
AXIS_LABEL_SIZE = 11
TICK_LABEL_SIZE = 9
LEGEND_TEXT_SIZE = 8.5
ANNOTATION_SIZE = 8.5

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": PANEL_TITLE_SIZE,
    "axes.labelsize": AXIS_LABEL_SIZE,
    "xtick.labelsize": TICK_LABEL_SIZE,
    "ytick.labelsize": TICK_LABEL_SIZE,
    "legend.fontsize": LEGEND_TEXT_SIZE,
    "text.color": TEXT_COLOR,
    "axes.labelcolor": TEXT_COLOR,
    "xtick.color": TEXT_COLOR,
    "ytick.color": TEXT_COLOR,
})

fig = plt.figure(figsize=(16, 12.2), facecolor=SURFACE)
gs = fig.add_gridspec(2, 2, width_ratios=[1.0, 1.12], height_ratios=[0.78, 1.25],
                      wspace=0.30, hspace=0.42)

# ---------- A: standardized protocol pressure ----------
ax = fig.add_subplot(gs[0, 0])
protocol_order = ["Patient", "Cluster", "Subtype"]
metric_cols = ["anti_leakage", "sequence_novelty", "subtype_shift", "auroc_stress"]
metric_labels = ["Anti-leakage", "Sequence\nnovelty", "Subtype\nshift", "AUROC\nstress"]
mat = stress_scaled.set_index("protocol_label").loc[protocol_order, metric_cols].values
im = ax.imshow(mat, cmap="YlGnBu", vmin=0, vmax=1, aspect="auto")
ax.set_yticks(np.arange(len(protocol_order)))
ax.set_yticklabels(protocol_order)
ax.set_xticks(np.arange(len(metric_cols)))
ax.set_xticklabels(metric_labels)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, f"{mat[i, j]:.2f}", ha="center", va="center", fontsize=ANNOTATION_SIZE,
                color=TEXT_COLOR if mat[i, j] < 0.62 else SURFACE,
                fontweight="bold" if mat[i, j] >= 0.74 else "normal")
for s in ("top", "right", "bottom", "left"):
    ax.spines[s].set_visible(False)
ax.tick_params(length=0)
ax.set_title("A  Standardized evaluation pressure", loc="left", fontweight="bold", color=TEXT_COLOR)
cbar = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.025)
cbar.set_label("percentile rank", fontsize=8.5)
cbar.ax.tick_params(labelsize=8)

# ---------- B: AUROC drop from Random ----------
ax = fig.add_subplot(gs[0, 1])
x = np.arange(len(STRICT_ORDER)); w = 0.23
for j, method in enumerate(METHODS):
    rows = drop_summary[drop_summary["method"] == method].set_index("protocol").loc[STRICT_ORDER]
    y = rows["mean_drop"].values
    yerr = np.vstack([y - rows["ci_low"].values, rows["ci_high"].values - y])
    xpos = x + (j - 1) * w
    ax.bar(xpos, y, w, color=M_COLOR[method], edgecolor=SURFACE, label=M_LABEL[method])
    ax.errorbar(xpos, y, yerr=yerr, fmt="none", ecolor=INK2, elinewidth=1.0, capsize=2.5)
    for xi, yi in zip(xpos, y):
        ax.text(xi, yi + 0.004, f"{yi:.3f}", ha="center", va="bottom", fontsize=8, color=TEXT_COLOR)
ax.axhline(0, color=INK2, lw=1)
ax.set_xticks(x)
ax.set_xticklabels([PROTO_LABEL[p] for p in STRICT_ORDER])
ax.set_ylabel("AUROC drop from Random")
ax.set_title("B  Performance drop under stricter protocols", loc="left", fontweight="bold", color=TEXT_COLOR)
leg = ax.legend(frameon=False, loc="upper left", ncol=1)
for txt in leg.get_texts():
    txt.set_color(TEXT_COLOR)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)

# ---------- C: subtype OOD method comparison ----------
ax = fig.add_subplot(gs[1, 0])
x = np.arange(len(METHODS))
rng = np.random.default_rng(42)
pivot_sub = subtype.pivot(index=["class", "drug"], columns="method", values="auroc").reset_index()
for _, g in pivot_sub.iterrows():
    ax.plot(x, [g[m] for m in METHODS], color=GRID, lw=0.7, alpha=0.45, zorder=1)
for j, method in enumerate(METHODS):
    vals = subtype.loc[subtype["method"] == method, "auroc"].values
    jitter = rng.normal(0, 0.035, len(vals))
    ax.scatter(np.full(len(vals), j) + jitter, vals, s=28, color=M_COLOR[method],
               alpha=0.34, edgecolor="none", zorder=2)
    mean, lo, hi = boot_ci(vals)
    ax.errorbar(j, mean, yerr=[[mean - lo], [hi - mean]], fmt="o", ms=10,
                color=M_COLOR[method], markeredgecolor=SURFACE, capsize=4, zorder=4)
    ax.text(j, hi + 0.008, f"{mean:.3f}", ha="center", va="bottom", fontsize=8.5, color=TEXT_COLOR)
ax.set_xticks(x)
ax.set_xticklabels([M_LABEL[m] for m in METHODS],  ha="center")
ax.set_ylabel("Subtype OOD AUROC")
ax.set_title("C  Model performance under Subtype OOD", loc="left", fontweight="bold", color=TEXT_COLOR)
ax.set_ylim(max(0.5, subtype["auroc"].min() - 0.04), min(1.02, subtype["auroc"].max() + 0.05))
for s in ("top", "right"):
    ax.spines[s].set_visible(False)

# ---------- D: Random CV minus Subtype OOD drop ----------
ax = fig.add_subplot(gs[1, 1])
plot_drop = drop_df.sort_values("drop", ascending=True).reset_index(drop=True)
y = np.arange(len(plot_drop))
colors = np.where(plot_drop["drop"] > 0.05, "#eb6834", "#b8b6b0")
ax.barh(y, plot_drop["drop"], color=colors, edgecolor=SURFACE, height=0.68)
ax.axvline(0, color=INK2, lw=1.0)
ax.axvline(0.05, color="#eb6834", lw=1.1, ls="--")
ax.text(0.052, len(plot_drop) - 0.1, "drop > 0.05", color="#eb6834", fontsize=8.5,
        ha="left", va="top")
for yi, r in plot_drop.iterrows():
    if r["drop"] > 0.05:
        ax.text(r["drop"] + 0.006, yi, f"{r['drug']} {r['drop']:+.3f}",
                va="center", ha="left", fontsize=7.6, color=TEXT_COLOR)
ax.set_yticks(y)
ax.set_yticklabels([f"{r['class']}/{r['drug']}" for _, r in plot_drop.iterrows()], fontsize=8)
ax.set_xlabel("AUROC drop: Random CV − Subtype OOD\n(ESM-nonlinear)")
ax.set_title("D  Random CV overestimates a subset of drug tasks", loc="left", fontweight="bold", color=TEXT_COLOR)
mean_drop = plot_drop["drop"].mean()
large_drop = int((plot_drop["drop"] > 0.05).sum())
stable = len(plot_drop) - large_drop
ax.text(0.82, 0.16,
        f"Mean drop = {mean_drop:.3f}\nStable/small change: {stable}/{len(plot_drop)}\nOverestimated: {large_drop}/{len(plot_drop)}",
        transform=ax.transAxes, ha="left", va="top", fontsize=8.5, color=TEXT_COLOR,
        bbox=dict(boxstyle="round,pad=0.35", facecolor=SURFACE, edgecolor=GRID, alpha=0.92))
for s in ("top", "right"):
    ax.spines[s].set_visible(False)

fig.tight_layout(rect=[0, 0, 1, 1])
fig.savefig(FIG_DIR / "fig3_combined.png", dpi=180, bbox_inches="tight", facecolor=SURFACE)
fig.savefig(FIG_DIR / "fig3_combined.pdf", bbox_inches="tight", facecolor=SURFACE)
print("saved", FIG_DIR / "fig3_combined.png")
print("saved", FIG_DIR / "fig3_combined.pdf")
plt.show()